编码方式：种类+回合

G：黄金数量

O：石油数量

S：钢铁数量

A：行动点数

P：战力（人口）数量

L：土地数量

注：无论哪个是最后一回合，最后一回合是没有谈判阶段的，需要跳过谈判阶段的代码！

## FUNCTION

### 土地加成情况

In [3]:
# Bonus情况
def bonus(land):
    g_bonus = 0
    o_bonus = 0
    s_bonus = 0
    for l in land:
        if l == 'A1':
            g_bonus += 3
        if l == 'A2':
            o_bonus += 2
        if l == 'A3':
            o_bonus += 5
        if l == 'A4':
            o_bonus += 3
        if l == 'A5':
            g_bonus += 5
        if l == 'A6':
            s_bonus += 3
        if l == 'A7':
            s_bonus += 3
        if l == 'A8':
            g_bonus += 3
        if l == 'B1':
            s_bonus += 3
        if l == 'B2':
            s_bonus += 4
        if l == 'B3':
            o_bonus += 3
        if l == 'B4':
            g_bonus += 5
        if l == 'C1':
            o_bonus += 4
        if l == 'C2':
            o_bonus += 3
        if l == 'C3':
            s_bonus += 4
        if l == 'C4':
            g_bonus += 3
        if l == 'D1':
            o_bonus += 5
        if l == 'D2':
            o_bonus += 6
        if l == 'D3':
            o_bonus += 3
        if l == 'D4':
            o_bonus += 3
        if l == 'E1':
            o_bonus += 3
        if l == 'E2':
            s_bonus += 3
        if l == 'E3':
            s_bonus += 4
        if l == 'E4':
            g_bonus += 5
        if l == 'F1':
            g_bonus += 2
        if l == 'F2':
            g_bonus += 5
        if l == 'F3':
            o_bonus += 3
        if l == 'F4':
            s_bonus += 4
        if l == 'G1':
            s_bonus += 3
        if l == 'G2':
            g_bonus += 4
    return g_bonus,o_bonus,s_bonus

### 战力部署

In [4]:
# 战力部署function（国家wise）
def deploy(land,country,total_power):
    while True:
        try:
            dict_deploy = {}
            total_people = 0
            for l in land:
                people = int(input(f'{country}部署在土地{l}上的战力数（不少于1）：'))
                if people < 1:
                    raise ValueError(f"地块 {l} 部署不足！已有领土必须至少保留1战力。")
                dict_deploy[l] = people
                total_people += people
            if total_people != total_power:
                raise ValueError(f"战力总数不对！{country}国分配了 {total_people} 点，但拥有 {total_power} 点。")
            
            return dict_deploy
        
        except ValueError as e:
            print(f"部署错误：{e}")
            print(f"请 {country} 国重新分配战力（确保每块地 >=1，总额应为 {total_power}）")

### 军事行动

In [5]:
# 开始行动
def military(t, order, clist, dict_land, dict_action, dict_oil, dict_steel, country_deploy, land_deploy, dict_ceasefire):
    print (f"\n === 第{t}回合：外交谈判环节 ===")
    for country in order:
        print(f"--- 当前行动国家：{country} ---")
        dict_country_move = {}
        while True:
            try:
                action = input(f'{country}选择行动（移动/进攻/停止）：',) #输入

                # -- 移动 --
                if action == '移动':
                    print(f"\n{country}国执行移动行动")
                    move_from = input('移出地块:(填入从哪块土地上移动出来的)').strip().upper()
                    move_to = input('目标地块:(填入移动到的编号)',).strip().upper()
                    
                    # 移出领土所有权校验
                    if move_from not in dict_land[country]:
                        print(f"错误：{move_from} 不属于 {country}，无法移出战力！")
                        continue

                    # 判定移动类型
                    is_sea = input('是否为跨海移动? (是/否): ').strip()
                    action_cost = 4 if is_sea == '是' else 2 # 跨海4点，普通2点
                    
                    # 确认剩余资源是否足够
                    if dict_action[country] < action_cost:
                        print(f"指令无效：剩余行动点数不足({dict_action[country]})")
                        continue

                    # 一旦指令有效，立刻扣除行动点数
                    dict_action[country] -= action_cost
                    print(f"{country}国扣除行动点数{action_cost}")

                    # 目标地块归属校验
                    target_owner = None
                    for c in clist:
                        if move_to in dict_land[c]:
                            target_owner = c
                            break
                    if target_owner is not None and target_owner != country:    # 如果目标地块有人占领且不是自己，移动失败
                        print(f"移动失败：土地 {move_to} 已被 {target_owner} 国占领！")
                        print("提示：移动只能在自家领土或无主土地进行。若要进入该地，请使用‘进攻’指令。")
                        continue
                    
                    # 判断移动范围
                    can_move = input('老师判断：是否在移动范围（1格）内（是/否）：',).strip()
                    if can_move == '否':
                        print("超出移动范围，行动失败")
                        continue

                    # 战力校验
                    current_people = country_deploy[country].get(move_from, 0)
                    if current_people <= 1:
                        print(f"错误：{move_from} 战力不足，必须保留至少1战力防守")
                        continue
                    
                    try:
                        move_ppl = int(input(f'移动战力数(当前{current_people}, 最大可移{min(5, current_people-1)})：',))
                        # 校验：移出地块是否有足够战力
                        if move_ppl > 5:
                            print("错误：单次移动不得超过 5 单位战力 ！")
                            continue
                        if move_ppl >= current_people:
                            print("错误：必须保留至少 1 战力在本国领土防守 ！")
                            continue
                    except ValueError:
                        print("输入错误，指令作废。")
                        continue


                    # 更新部署数据
                    country_deploy[country][move_to] = country_deploy[country].get(move_to, 0) + move_ppl
                    country_deploy[country][move_from] -= move_ppl
                    if move_to not in dict_land[country]:
                        dict_land[country].append(move_to)
                        print(f"{country}国成功占领/移动{move_ppl}战力至新领土：{move_to}")
                    
                    land_deploy[move_from] = country_deploy[country][move_from]
                    land_deploy[move_to] = country_deploy[country][move_to]

                
                # -- 进攻 --
                if action == '进攻':
                    print(f"\n{country}国发起进攻")
                    
                    attack_from = input('发起进攻的地块: ').strip().upper()
                    # 确定发起进攻的地块是否属于该国
                    if attack_from not in dict_land[country]:
                        print(f"错误：{attack_from} 不属于 {country}，无法从该地发起进攻！")
                        continue

                    target_land = input('目标地块: ').strip().upper()
                    # 确定没有进攻自己的领土
                    if target_land in dict_land[country]:
                        print(f"错误：{target_land} 已经是你的领土了，请重新选择目标！")
                        continue

                    # 判断进攻类型
                    is_sea = input('是否为跨海进攻? (是/否): ').strip()
                    action_cost = 2 if is_sea == '是' else 1
                    oil_cost = 4 if is_sea == '是' else 2
                    steel_cost = 2 if is_sea == '是' else 1

                    # 校验资源是否足够 (若不够则无法发起行动，不扣点)
                    if dict_action[country] < action_cost or dict_oil[country] < oil_cost or dict_steel[country] < steel_cost:
                        print("资源不足，无法发起进攻！")
                        continue

                    # 校验发起地战力是否足够
                    current_people = country_deploy[country].get(attack_from, 0)
                    if current_people <= 1:
                        print(f"{attack_from} 战力不足以发起进攻（需保留1人防守）。")
                        continue
                    try:
                        attack_ppl = int(input(f'移动战力数(当前{current_people}, 最大可移{min(5, current_people-1)})：',))
                        if attack_ppl >= current_people:
                            print("必须保留至少 1 战力在本国领土防守！")
                            continue
                    except ValueError:
                        print('输入错误，指令作废')
                        continue

                    # 一旦指令有效，立刻进行资源扣除
                    dict_action[country] -= action_cost
                    dict_oil[country] -= oil_cost
                    dict_steel[country] -= steel_cost
                    print(f"{country}国扣除行动点数{action_cost}, 石油{oil_cost}, 钢铁{steel_cost}")

                    # 判断是否在攻击范围（1格）以内，即使判定为否，资源已经扣除了
                    attack_in_range = input('老师判断：是否在攻击范围（1格）内? (是/否):').strip()
                    if attack_in_range == '否':
                        print(f"判定结果：目标超出了攻击范围！{country} 损失了资源，但行动失败。")
                        continue

                    # 获取防守方信息
                    defender = None
                    for c in clist:
                        if target_land in dict_land[c]:
                            defender = c
                            break
                    # 校验停战状态 
                    if defender:
                        pair = tuple(sorted([country, defender]))
                        pair_key = f"{pair[0]}-{pair[1]}"
                        if pair_key in dict_ceasefire:
                            if t <= dict_ceasefire[pair_key]:
                                print(f"警告：{country} 与 {defender} 处于强制停战期（至第 {dict_ceasefire[pair_key]} 回合）！")
                                print("根据手册规则，停战协定强制生效，行动撤回。")
                                continue
                    # 如果 target_land 不在字典里，说明是无主土地或输入错误，默认防守力0
                    defend_ppl = land_deploy.get(target_land, 0)
                    print(f"战斗：进攻方 {attack_ppl} vs 防守方 {defend_ppl}")
                    

                    # 结果判定
                    # 进攻成功
                    if attack_ppl > defend_ppl:
                        print(f"进攻成功！{country}占领了 {target_land}")
                        # 更新防守方领土情况
                        for c in clist:
                            if target_land in dict_land[c]:
                                dict_land[c].remove(target_land)
                                country_deploy[c].pop(target_land, None)
                        print(f"原领主{c}国已失去该领土")
                        # 更新进攻方领土情况
                        dict_land[country].append(target_land)
                        country_deploy[country][target_land] = attack_ppl
                        country_deploy[country][attack_from] -= attack_ppl

                    # 进攻失败
                    elif defend_ppl > attack_ppl:
                        print(f"进攻失败！{country}损失了全部进攻战力（{attack_ppl}人）")
                        country_deploy[country][attack_from] -= attack_ppl
                    
                    # 平局
                    else:
                        print("平局！双方均无损失，进攻部队退回原籍")
                    
                    # 同步全局地块分布
                    land_deploy[attack_from] = country_deploy[country][attack_from]
                    land_deploy[target_land] = land_deploy.get(target_land, 0) if defend_ppl >= attack_ppl else attack_ppl


                # -- 不行动 --
                if action == '停止':
                    print(f"--- {country} 行动结束 ---\n")
                    break


                # -- 指令输错 --
                if action not in ['移动','进攻','停止']:
                    raise ValueError("指令错误，请重新输入")
                
            except ValueError as e:
                print(f"{e}\n 请重新输入有效指令\n")       

### 谈判阶段

In [6]:
# T1谈判情况
def diplomacy(t, dict_gold, dict_oil, dict_steel, dict_ceasefire):
    print(f"\n === 第{t}回合：外交谈判环节 ===")
    
    while True:
        action = input("请输入谈判类型 (贸易/停战/查看/结束): ").strip()
        
        if action == '结束':
            break


        if action == '查看':
            print(f"当前生效的停战协议: {dict_ceasefire}")
            continue


        # --- 贸易协定 ---
        if action == '贸易':
            country_a = input("国家1: ").strip().upper()
            country_b = input("国家2: ").strip().upper()
            
            print(f"请输入 {country_a} 给出的资源(填数字，若无则填0):")
            a_gold = int(input(f"{country_a} 给出黄金: "))
            a_oil = int(input(f"{country_a} 给出石油: "))
            a_steel = int(input(f"{country_a} 给出钢铁: "))

            print(f"请输入 {country_b} 给出的资源(填数字，若无则填0):")
            b_gold = int(input(f"{country_b} 给出黄金: "))
            b_oil = int(input(f"{country_b} 给出石油: "))
            b_steel = int(input(f"{country_b} 给出钢铁: "))

            # 校验资源余额
            if dict_gold[country_a] < a_gold or dict_oil[country_a] < a_oil or dict_steel[country_a] < a_steel:
                print(f'交易失败：{country_a}国资源余额不足！')
                continue
            if dict_gold[country_b] < b_gold or dict_oil[country_b] < b_oil or dict_steel[country_b] < b_steel:
                print(f'交易失败：{country_b}国资源余额不足！')
                continue

            # 执行交换
            dict_gold[country_a] += (b_gold - a_gold)
            dict_oil[country_a] += (b_oil - a_oil)
            dict_steel[country_a] += (b_steel - a_steel)

            dict_gold[country_b] += (a_gold - b_gold)
            dict_oil[country_b] += (a_oil - b_oil)
            dict_steel[country_b] += (a_steel - b_steel)

            print(f"贸易成功！{country_a} 与 {country_b} 已完成资源互换。")


        # --- 停战协定 ---
        elif action == '停战':
            c1 = input("国家1: ").strip().upper()
            c2 = input("国家2: ").strip().upper()
            
            pair = tuple(sorted([c1, c2]))  # 排序以保证 A-B 和 B-A 在字典中键名一致
            pair_key = f"{pair[0]}-{pair[1]}"
            
            # 停战协定承诺未来3回合内不发起进攻 
            expire_t = t + 3
            dict_ceasefire[pair_key] = expire_t
            
            print(f"停战协定已生效！{c1} 与 {c2} 在第{t+1}到第{expire_t}回合内不得互相进攻。")

### 行动点数兑换

In [7]:
def exchange_action(t,clist,dict_action,dict_gold,dict_oil,dict_steel):
    print(f"\n === 第 {t} 回合：行动点数结算与兑换 ===")
    
    for country in clist:
        print (f'--- {country}国结算行动点数 ---')
        rem_action = dict_action[country]
        if rem_action <= 0:
            print(f"账户提示：{country} 国行动点已耗尽，无需兑换。")
            continue
        
        print(f"\n{country}拥有{rem_action}点剩余行动点")
        print(f"兑换比例：1行动点数 = 10黄金 / 8石油 / 5钢铁")
        
        while rem_action > 0:
            try:
                exchange_type = input("请选择兑换目标 (黄金/石油/钢铁/放弃): ").strip()
                
                if exchange_type == '放弃':
                    print(f"{country} 放弃了剩余 AP 的兑换。")
                    break
                
                if exchange_type not in ['黄金', '石油', '钢铁']:
                    print("输入无效，请输入：黄金、石油或钢铁。")
                    continue
                
                num_to_exchange = int(input(f"兑换多少个行动点数到{exchange_type}? : "))
                
                if num_to_exchange > rem_action or num_to_exchange <= 0:
                    print(f"数量错误！{country}国只有{rem_action}个行动点数。")
                    continue
                
                # 执行兑换逻辑
                if exchange_type == '黄金':
                    g_earned = num_to_exchange * 10
                    dict_gold[country] += g_earned
                    print(f"兑换成功：获得 {g_earned} 黄金")
                elif exchange_type == '石油':
                    o_earned = num_to_exchange * 8
                    dict_oil[country] += o_earned
                    print(f"兑换成功：获得 {o_earned} 石油")
                elif exchange_type == '钢铁':
                    s_earned = num_to_exchange * 5
                    dict_steel[country] += s_earned
                    print(f"兑换成功：获得 {s_earned} 钢铁")
                
                rem_action -= num_to_exchange
                
            except ValueError:
                print("请输入有效的数字")
        
        # 兑换完成后，该国行动点数清零 
        dict_action[country] = 0
        print(f"{country} 国结算完毕\n")

### 保存和读取数据

In [2]:
import json
import os

# 定义文件名
filename = "gss_save_data.json"

# 抓取快照
def get_current_snapshot(t):
    return {
        "round": t,
        "land": dict_land.copy(),
        "gold": dict_gold.copy(),
        "oil": dict_oil.copy(),
        "steel": dict_steel.copy(),
        "people": dict_people.copy(),
        "action": dict_action.copy(),
        "total": dict_total.copy(),
        "ceasefire": dict_ceasefire.copy()
    }

# 保存到本地JSON文件
def save_gss(t):
    history = []
    if os.path.exists(filename):
        with open(filename, 'r', encoding='utf-8') as f:
            history = json.load(f)
    new_snapshot = get_current_snapshot(t)  #保存快照
    history.append(new_snapshot)
    with open(filename, 'w', encoding='utf-8') as f: #写入
        json.dump(history, f, ensure_ascii=False, indent=4)
    print(f"第{t}回合数据已存至本地。")

# 读取本地JSON文件
def load_gss():
    global dict_land, dict_gold, dict_oil, dict_steel, dict_people, dict_action, dict_total, dict_ceasefire
    if not os.path.exists(filename):
        print("未发现存档文件")
        return 0
    
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            history = json.load(f)
        
        if not history: return 0
        last_state = history[-1]    # 获取最后一条记录
        
        # 批量还原dictionary数据
        dict_land.update(last_state.get('land', {}))
        dict_gold.update(last_state.get('gold', {}))
        dict_oil.update(last_state.get('oil', {}))
        dict_steel.update(last_state.get('steel', {}))
        dict_people.update(last_state.get('people', {}))
        dict_action.update(last_state.get('action', {}))
        dict_total.update(last_state.get('total', {}))
        dict_ceasefire.update(last_state.get('ceasefire', {}))
        
        print(f"已成功读取进度！当前处于第 {last_state['round']} 回合结束状态。")
        return last_state['round']
        
    except Exception as e:
        print(f"读取存档失败: {e}")
        return 0

## 第0回合

In [8]:
t=0

In [9]:
clist = input("国家名称（用英文逗号隔开）：")
clist = [x.strip().upper() for x in clist.split(',') if x.strip()]

gss_history = []   # 总历史记录列表

# 所有dictionary
dict_land = {}
dict_gold = {}
dict_oil = {}
dict_steel = {}
dict_people = {}
dict_action = {}
dict_total = {}
dict_ceasefire = {}

In [ ]:
print("=== GSS 全球战略模拟：第 0 回合土地购买 ===\n")

for country in clist:
    print(f'--- {country}国土地购买情况 ---')
    G0 = 30
    O0 = 0
    S0 = 0
    P0 = 0
    A0 = 0
    while True:
        try:
            land = input("请输入土地编号（用英文逗号隔开）：")
            land = [x.strip().upper() for x in land.split(',') if x.strip()]
            L0 = len(land)
            # 校验：不能超过 3 块地（30 黄金上限） 
            if L0 > 3:
                raise ValueError(f"资金不足！每块地10黄金，你只有30黄金，最多买3块。")
            break
        except ValueError as e:
            print(f"{e}\n请重新输入")

    G_0 = G0-10*L0
    T0 = G_0 + O0*10/8 + S0*10/5 + A0*10 + len(land)*20
    print(f'{country}国第0回合占有的土地为:',land)
    print(f'{country}国第0回合土地总数为:',L0)
    print('\n')
    dict_land[country] = land
    dict_gold[country] = G_0
    dict_oil[country] = O0
    dict_steel[country] = S0
    dict_people[country] = P0
    dict_action[country] = A0
    dict_total[country] = T0
    
print('\n'+"="*30)
print('第0回合结束时各国土地购买概况：',dict_land)
print('第0回合结束时各国剩余黄金概况：',dict_gold)
print('第0回合结束时各国剩余石油概况：',dict_oil)
print('第0回合结束时各国剩余钢铁概况：',dict_steel)
print('第0回合结束时各国战力概况：',dict_people)
print('第0回合结束时各国剩余行动点数概况：',dict_action)
print('第0回合结束时各国资源换算成黄金总量情况：',dict_total)

=== GSS 全球战略模拟：第 0 回合土地购买 ===

--- A国土地购买情况 ---
A国第0回合占有的土地为: ['A1', 'A2']
A国第0回合土地总数为: 2


--- B国土地购买情况 ---
B国第0回合占有的土地为: ['B1', 'B2']
B国第0回合土地总数为: 2


--- C国土地购买情况 ---
C国第0回合占有的土地为: ['C1', 'C2']
C国第0回合土地总数为: 2


--- D国土地购买情况 ---
D国第0回合占有的土地为: ['D1', 'D2']
D国第0回合土地总数为: 2



第0回合结束时各国土地购买概况： {'A': ['A1', 'A2'], 'B': ['B1', 'B2'], 'C': ['C1', 'C2'], 'D': ['D1', 'D2']}
第0回合结束时各国剩余黄金概况： {'A': 10, 'B': 10, 'C': 10, 'D': 10}
第0回合结束时各国剩余石油概况： {'A': 0, 'B': 0, 'C': 0, 'D': 0}
第0回合结束时各国剩余钢铁概况： {'A': 0, 'B': 0, 'C': 0, 'D': 0}
第0回合结束时各国战力概况： {'A': 0, 'B': 0, 'C': 0, 'D': 0}
第0回合结束时各国剩余行动点数概况： {'A': 0, 'B': 0, 'C': 0, 'D': 0}
第0回合结束时各国资源换算成黄金总量情况： {'A': 50.0, 'B': 50.0, 'C': 50.0, 'D': 50.0}


### 保存本回合所有资源情况

In [11]:
save_gss(t)

第0回合数据已存至本地。


## 第1回合

### 读取上一回合情况

In [12]:
load_gss()
t = 1

已成功读取进度！当前处于第 0 回合结束状态。


### 回合开始阶段

In [13]:
for country in clist:
    land = dict_land[country]
    g_bonus,o_bonus,s_bonus = bonus(land)
    dict_gold[country] = dict_gold[country] + g_bonus
    dict_oil[country] = len(land)*2 + o_bonus
    dict_steel[country] = len(land) + s_bonus
    dict_action[country] = 2
    dict_people[country] = 10

print(f'第{t}回合开始时各国土地购买概况：',dict_land)
print(f'第{t}回合开始时各国剩余黄金概况：',dict_gold)
print(f'第{t}回合开始时各国剩余石油概况：',dict_oil)
print(f'第{t}回合开始时各国剩余钢铁概况：',dict_steel)
print(f'第{t}回合开始时各国战力概况：',dict_people)
print(f'第{t}回合开始时各国剩余行动点数概况：',dict_action)

第1回合开始时各国土地购买概况： {'A': ['A1', 'A2'], 'B': ['B1', 'B2'], 'C': ['C1', 'C2'], 'D': ['D1', 'D2']}
第1回合开始时各国剩余黄金概况： {'A': 13, 'B': 10, 'C': 10, 'D': 10}
第1回合开始时各国剩余石油概况： {'A': 6, 'B': 4, 'C': 11, 'D': 15}
第1回合开始时各国剩余钢铁概况： {'A': 2, 'B': 9, 'C': 2, 'D': 2}
第1回合开始时各国战力概况： {'A': 10, 'B': 10, 'C': 10, 'D': 10}
第1回合开始时各国剩余行动点数概况： {'A': 2, 'B': 2, 'C': 2, 'D': 2}


### 战力部署阶段

In [ ]:
country_deploy = {}
land_deploy = {}
for country in clist:
    current_res = deploy(dict_land[country],country,dict_people[country])
    country_deploy[country] = current_res
    land_deploy.update(current_res)
    print(f"{country} 战力部署成功\n")

print('\n' + '='*30)
print(f'第{t}回合各国土地部署情况:',country_deploy)
print(f'第{t}回合全球战力分布情况:',land_deploy)

A 战力部署成功

B 战力部署成功

C 战力部署成功

D 战力部署成功


------------------------------
第1回合各国土地部署情况: {'A': {'A1': 5, 'A2': 5}, 'B': {'B1': 4, 'B2': 6}, 'C': {'C1': 3, 'C2': 7}, 'D': {'D1': 2, 'D2': 8}}
第1回合全球战力分布情况: {'A1': 5, 'A2': 5, 'B1': 4, 'B2': 6, 'C1': 3, 'C2': 7, 'D1': 2, 'D2': 8}


### 行动阶段

In [15]:
# 抽签决定行动顺序
order = input('输入抽签的行动顺序：(用英文逗号隔开)',)
order = [x.strip().upper() for x in order.split(',') if x.strip()]

In [16]:
# 执行行动function
military(
    t=t, 
    order=order, 
    clist=clist, 
    dict_land=dict_land, 
    dict_action=dict_action, 
    dict_oil=dict_oil, 
    dict_steel=dict_steel, 
    country_deploy=country_deploy, 
    land_deploy=land_deploy, 
    dict_ceasefire=dict_ceasefire
)


 === 第1回合：外交谈判环节 ===
--- 当前行动国家：A ---

A国执行移动行动
A国扣除行动点数2
A国成功占领/移动4战力至新领土：A3
--- A 行动结束 ---

--- 当前行动国家：B ---

B国发起进攻
B国扣除行动点数1, 石油2, 钢铁1
战斗：进攻方 3 vs 防守方 3
平局！双方均无损失，进攻部队退回原籍
--- B 行动结束 ---

--- 当前行动国家：C ---

C国发起进攻
C国扣除行动点数1, 石油2, 钢铁1
战斗：进攻方 2 vs 防守方 4
进攻失败！C损失了全部进攻战力（2人）
--- C 行动结束 ---

--- 当前行动国家：D ---

D国执行移动行动
指令无效：剩余行动点数不足(2)

D国执行移动行动
D国扣除行动点数2
D国成功占领/移动1战力至新领土：E1
--- D 行动结束 ---



### 谈判阶段

In [17]:
#执行谈判function
diplomacy(
    t=t, 
    dict_gold=dict_gold, 
    dict_oil=dict_oil,
    dict_steel=dict_steel,
    dict_ceasefire=dict_ceasefire)


 === 第1回合：外交谈判环节 ===
停战协定已生效！A 与 B 在第2到第4回合内不得互相进攻。
请输入 C 给出的资源(填数字，若无则填0):
请输入 D 给出的资源(填数字，若无则填0):
交易失败：D国资源余额不足！
请输入 C 给出的资源(填数字，若无则填0):
请输入 D 给出的资源(填数字，若无则填0):
交易失败：D国资源余额不足！


### 剩余行动点数兑换阶段

In [18]:
# 执行行动点数兑换function
exchange_action(
    t=t,
    clist=clist,
    dict_action=dict_action,
    dict_gold=dict_gold,
    dict_oil=dict_oil,
    dict_steel=dict_steel)


 === 第 1 回合：行动点数结算与兑换 ===
--- A国结算行动点数 ---
账户提示：A 国行动点已耗尽，无需兑换。
--- B国结算行动点数 ---

B拥有1点剩余行动点
兑换比例：1行动点数 = 10黄金 / 8石油 / 5钢铁
兑换成功：获得 10 黄金
B 国结算完毕

--- C国结算行动点数 ---

C拥有1点剩余行动点
兑换比例：1行动点数 = 10黄金 / 8石油 / 5钢铁
兑换成功：获得 8 石油
C 国结算完毕

--- D国结算行动点数 ---
账户提示：D 国行动点已耗尽，无需兑换。


### 回合结束结算阶段

In [19]:
# 换算成黄金总量
for country in clist:
    dict_total[country] = dict_gold[country] + dict_oil[country]*10/8 + dict_steel[country]*10/5 + dict_action[country]*10 + len(dict_land[country])*20

# 回合结束后的资源概况
print("\n" + "="*30)
print(f'第{t}回合结束时各国土地概况：',dict_land)
print(f'第{t}回合结束时各国剩余黄金概况：',dict_gold)
print(f'第{t}回合结束时各国剩余石油概况：',dict_oil)
print(f'第{t}回合结束时各国剩余钢铁概况：',dict_steel)
print(f'第{t}回合结束时各国战力概况：',dict_people)
print(f'第{t}回合结束时各国剩余行动点数概况：',dict_action)
print(f'第{t}回合结束时各国资源换算成黄金总量情况：',dict_total)


第1回合结束时各国土地概况： {'A': ['A1', 'A2', 'A3'], 'B': ['B1', 'B2'], 'C': ['C1', 'C2'], 'D': ['D1', 'D2', 'E1']}
第1回合结束时各国剩余黄金概况： {'A': 13, 'B': 20, 'C': 10, 'D': 10}
第1回合结束时各国剩余石油概况： {'A': 6, 'B': 2, 'C': 17, 'D': 15}
第1回合结束时各国剩余钢铁概况： {'A': 2, 'B': 8, 'C': 1, 'D': 2}
第1回合结束时各国战力概况： {'A': 10, 'B': 10, 'C': 10, 'D': 10}
第1回合结束时各国剩余行动点数概况： {'A': 0, 'B': 0, 'C': 0, 'D': 0}
第1回合结束时各国资源换算成黄金总量情况： {'A': 84.5, 'B': 78.5, 'C': 73.25, 'D': 92.75}


### 保存本回合所有资源情况

In [20]:
save_gss(t)

第1回合数据已存至本地。


## 第2回合

In [21]:
load_gss()
t = 2

已成功读取进度！当前处于第 1 回合结束状态。


### 回合开始阶段

In [22]:
for country in clist:
    land = dict_land[country]
    g_bonus,o_bonus,s_bonus = bonus(land)
    dict_gold[country] = dict_gold[country] + g_bonus + dict_people[country]    #新增黄金数=bonus+战力数
    dict_oil[country] = len(land)*2 + o_bonus
    dict_steel[country] = len(land) + s_bonus
    dict_action[country] = len(land)
    if dict_people[country] >= 10:
        dict_people[country] += 1
    else:
        dict_people[country] = int((10-dict_people[country])*0.5+2)

print("\n" + "="*30)
print(f'第{t}回合开始时各国土地购买概况：',dict_land)
print(f'第{t}回合开始时各国剩余黄金概况：',dict_gold)
print(f'第{t}回合开始时各国剩余石油概况：',dict_oil)
print(f'第{t}回合开始时各国剩余钢铁概况：',dict_steel)
print(f'第{t}回合开始时各国战力概况：',dict_people)
print(f'第{t}回合开始时各国剩余行动点数概况：',dict_action)


第2回合开始时各国土地购买概况： {'A': ['A1', 'A2', 'A3'], 'B': ['B1', 'B2'], 'C': ['C1', 'C2'], 'D': ['D1', 'D2', 'E1']}
第2回合开始时各国剩余黄金概况： {'A': 26, 'B': 30, 'C': 20, 'D': 20}
第2回合开始时各国剩余石油概况： {'A': 13, 'B': 4, 'C': 11, 'D': 20}
第2回合开始时各国剩余钢铁概况： {'A': 3, 'B': 9, 'C': 2, 'D': 3}
第2回合开始时各国战力概况： {'A': 11, 'B': 11, 'C': 11, 'D': 11}
第2回合开始时各国剩余行动点数概况： {'A': 3, 'B': 2, 'C': 2, 'D': 3}


### 战力部署阶段

In [23]:
country_deploy = {}
land_deploy = {}
for country in clist:
    current_res = deploy(dict_land[country],country,dict_people[country])
    country_deploy[country] = current_res
    land_deploy.update(current_res)
    print(f"{country} 战力部署成功\n")

print('\n' + '-'*30)
print(f'第{t}回合各国土地部署情况:',country_deploy)
print(f'第{t}回合全球战力分布情况:',land_deploy)

部署错误：战力总数不对！A国分配了 10 点，但拥有 11 点。
请 A 国重新分配战力（确保每块地 >=1，总额应为 11）
A 战力部署成功

部署错误：战力总数不对！B国分配了 10 点，但拥有 11 点。
请 B 国重新分配战力（确保每块地 >=1，总额应为 11）
B 战力部署成功

C 战力部署成功

部署错误：战力总数不对！D国分配了 10 点，但拥有 11 点。
请 D 国重新分配战力（确保每块地 >=1，总额应为 11）
部署错误：战力总数不对！D国分配了 12 点，但拥有 11 点。
请 D 国重新分配战力（确保每块地 >=1，总额应为 11）
D 战力部署成功


------------------------------
第2回合各国土地部署情况: {'A': {'A1': 2, 'A2': 4, 'A3': 5}, 'B': {'B1': 4, 'B2': 7}, 'C': {'C1': 6, 'C2': 5}, 'D': {'D1': 3, 'D2': 4, 'E1': 4}}
第2回合全球战力分布情况: {'A1': 2, 'A2': 4, 'A3': 5, 'B1': 4, 'B2': 7, 'C1': 6, 'C2': 5, 'D1': 3, 'D2': 4, 'E1': 4}


### 行动阶段

In [24]:
# 抽签决定行动顺序
order = input('输入抽签的行动顺序：(用英文逗号隔开)',)
order = [x.strip().upper() for x in order.split(',') if x.strip()]

In [25]:
# 执行行动function
military(
    t=t, 
    order=order, 
    clist=clist, 
    dict_land=dict_land, 
    dict_action=dict_action, 
    dict_oil=dict_oil, 
    dict_steel=dict_steel, 
    country_deploy=country_deploy, 
    land_deploy=land_deploy, 
    dict_ceasefire=dict_ceasefire
)


 === 第2回合：外交谈判环节 ===
--- 当前行动国家：B ---

B国发起进攻
错误：E1 不属于 B，无法从该地发起进攻！

B国发起进攻
B国扣除行动点数2, 石油4, 钢铁2
战斗：进攻方 3 vs 防守方 4
进攻失败！B损失了全部进攻战力（3人）
--- B 行动结束 ---

--- 当前行动国家：C ---

C国执行移动行动
C国扣除行动点数2
C国成功占领/移动5战力至新领土：C4
--- C 行动结束 ---

--- 当前行动国家：A ---
--- A 行动结束 ---

--- 当前行动国家：D ---
--- D 行动结束 ---



### 谈判阶段

In [ ]:
#执行谈判function
diplomacy(
    t=t, 
    dict_gold=dict_gold, 
    dict_oil=dict_oil,
    dict_steel=dict_steel,
    dict_ceasefire=dict_ceasefire)

### 剩余行动点数兑换阶段

In [ ]:
# 执行行动点数兑换function
exchange_action(
    t=t,
    clist=clist,
    dict_action=dict_action,
    dict_gold=dict_gold,
    dict_oil=dict_oil,
    dict_steel=dict_steel)

### 回合结束结算阶段

In [ ]:
# 换算成黄金总量
for country in clist:
    dict_total[country] = dict_gold[country] + dict_oil[country]*10/8 + dict_steel[country]*10/5 + dict_action[country]*10 + len(dict_land[country])*20

# 回合结束后的资源概况
print("\n" + "="*30)
print(f'第{t}回合结束时各国土地购买概况：',dict_land)
print(f'第{t}回合结束时各国剩余黄金概况：',dict_gold)
print(f'第{t}回合结束时各国剩余石油概况：',dict_oil)
print(f'第{t}回合结束时各国剩余钢铁概况：',dict_steel)
print(f'第{t}回合结束时各国战力概况：',dict_people)
print(f'第{t}回合结束时各国剩余行动点数概况：',dict_action)
print(f'第{t}回合结束时各国资源换算成黄金总量情况：',dict_total)

### 保存本回合所有资源情况

In [ ]:
save_gss(t)

## 第3回合

In [ ]:
load_gss()
t = 3

### 回合开始阶段

In [ ]:
for country in clist:
    land = dict_land[country]
    g_bonus,o_bonus,s_bonus = bonus(land)
    dict_gold[country] = dict_gold[country] + g_bonus + dict_people[country]
    dict_oil[country] = len(land)*2 + o_bonus
    dict_steel[country] = len(land) + s_bonus
    dict_action[country] = len(land)
    if dict_people[country] >= 10:
        dict_people[country] += 1
    else:
        dict_people[country] = int((10-dict_people[country])*0.5+2)

print(f'第{t}回合开始时各国土地概况：',dict_land)
print(f'第{t}回合开始时各国剩余黄金概况：',dict_gold)
print(f'第{t}回合开始时各国剩余石油概况：',dict_oil)
print(f'第{t}回合开始时各国剩余钢铁概况：',dict_steel)
print(f'第{t}回合开始时各国战力概况：',dict_people)
print(f'第{t}回合开始时各国剩余行动点数概况：',dict_action)

### 战力部署阶段

In [ ]:
country_deploy = {}
land_deploy = {}
for country in clist:
    current_res = deploy(dict_land[country],country,dict_people[country])
    country_deploy[country] = current_res
    land_deploy.update(current_res)
    print(f"{country} 战力部署成功")

print('\n' + '-'*30)
print(f'第{t}回合各国土地部署情况:',country_deploy)
print(f'第{t}回合全球战力分布情况:',land_deploy)

### 行动阶段

In [ ]:
# 抽签决定行动顺序
order = input('输入抽签的行动顺序：(用英文逗号隔开)',)
order = [x.strip().upper() for x in order.split(',') if x.strip()]

In [ ]:
# 执行行动function
military(
    t=t, 
    order=order, 
    clist=clist, 
    dict_land=dict_land, 
    dict_action=dict_action, 
    dict_oil=dict_oil, 
    dict_steel=dict_steel, 
    country_deploy=country_deploy, 
    land_deploy=land_deploy, 
    dict_ceasefire=dict_ceasefire
)

### 谈判阶段

In [ ]:
#执行谈判function
diplomacy(
    t=t, 
    dict_gold=dict_gold, 
    dict_oil=dict_oil,
    dict_steel=dict_steel,
    dict_ceasefire=dict_ceasefire)

### 剩余行动点数兑换阶段

In [ ]:
# 执行行动点数兑换function
exchange_action(
    t=t,
    clist=clist,
    dict_action=dict_action,
    dict_gold=dict_gold,
    dict_oil=dict_oil,
    dict_steel=dict_steel)

### 回合结束结算阶段

In [ ]:
# 换算成黄金总量
for country in clist:
    dict_total[country] = dict_gold[country] + dict_oil[country]*10/8 + dict_steel[country]*10/5 + dict_action[country]*10 + len(dict_land[country])*20

# 回合结束后的资源概况
print("\n" + "="*30)
print(f'第{t}回合结束时各国土地概况：',dict_land)
print(f'第{t}回合结束时各国剩余黄金概况：',dict_gold)
print(f'第{t}回合结束时各国剩余石油概况：',dict_oil)
print(f'第{t}回合结束时各国剩余钢铁概况：',dict_steel)
print(f'第{t}回合结束时各国战力概况：',dict_people)
print(f'第{t}回合结束时各国剩余行动点数概况：',dict_action)
print(f'第{t}回合结束时各国资源换算成黄金总量情况：',dict_total)

### 保存本回合所有资源情况

In [ ]:
save_gss(t)

## 第4回合

In [ ]:
load_gss()
t = 4

### 回合开始阶段

In [ ]:
for country in clist:
    land = dict_land[country]
    g_bonus,o_bonus,s_bonus = bonus(land)
    dict_gold[country] = dict_gold[country] + g_bonus + dict_people[country]
    dict_oil[country] = len(land)*2 + o_bonus
    dict_steel[country] = len(land) + s_bonus
    dict_action[country] = len(land)
    if dict_people[country] >= 10:
        dict_people[country] += 1
    else:
        dict_people[country] = int((10-dict_people[country])*0.5+2)

print(f'第{t}回合开始时各国土地概况：',dict_land)
print(f'第{t}回合开始时各国剩余黄金概况：',dict_gold)
print(f'第{t}回合开始时各国剩余石油概况：',dict_oil)
print(f'第{t}回合开始时各国剩余钢铁概况：',dict_steel)
print(f'第{t}回合开始时各国战力概况：',dict_people)
print(f'第{t}回合开始时各国剩余行动点数概况：',dict_action)

### 战力部署阶段

In [ ]:
country_deploy = {}
land_deploy = {}
for country in clist:
    current_res = deploy(dict_land[country],country,dict_people[country])
    country_deploy[country] = current_res
    land_deploy.update(current_res)
    print(f"{country} 战力部署成功")

print('\n' + '-'*30)
print(f'第{t}回合各国土地部署情况:',country_deploy)
print(f'第{t}回合全球战力分布情况:',land_deploy)

### 行动阶段

In [ ]:
# 抽签决定行动顺序
order = input('输入抽签的行动顺序：(用英文逗号隔开)',)
order = [x.strip().upper() for x in order.split(',') if x.strip()]

In [ ]:
# 执行行动function
military(
    t=t, 
    order=order, 
    clist=clist, 
    dict_land=dict_land, 
    dict_action=dict_action, 
    dict_oil=dict_oil, 
    dict_steel=dict_steel, 
    country_deploy=country_deploy, 
    land_deploy=land_deploy, 
    dict_ceasefire=dict_ceasefire
)

### 谈判阶段

In [ ]:
#执行谈判function
diplomacy(
    t=t, 
    dict_gold=dict_gold, 
    dict_oil=dict_oil,
    dict_steel=dict_steel,
    dict_ceasefire=dict_ceasefire)

### 剩余行动点数兑换阶段

In [ ]:
# 执行行动点数兑换function
exchange_action(
    t=t,
    clist=clist,
    dict_action=dict_action,
    dict_gold=dict_gold,
    dict_oil=dict_oil,
    dict_steel=dict_steel)

### 回合结束结算阶段

In [ ]:
# 换算成黄金总量
for country in clist:
    dict_total[country] = dict_gold[country] + dict_oil[country]*10/8 + dict_steel[country]*10/5 + dict_action[country]*10 + len(dict_land[country])*20

# 回合结束后的资源概况
print("\n" + "="*30)
print(f'第{t}回合结束时各国土地概况：',dict_land)
print(f'第{t}回合结束时各国剩余黄金概况：',dict_gold)
print(f'第{t}回合结束时各国剩余石油概况：',dict_oil)
print(f'第{t}回合结束时各国剩余钢铁概况：',dict_steel)
print(f'第{t}回合结束时各国战力概况：',dict_people)
print(f'第{t}回合结束时各国剩余行动点数概况：',dict_action)
print(f'第{t}回合结束时各国资源换算成黄金总量情况：',dict_total)

### 保存本回合所有资源情况

In [ ]:
save_gss(t)

## 第5回合

In [ ]:
load_gss
t = 5

### 回合开始阶段

In [ ]:
for country in clist:
    land = dict_land[country]
    g_bonus,o_bonus,s_bonus = bonus(land)
    dict_gold[country] = dict_gold[country] + g_bonus + dict_people[country]
    dict_oil[country] = len(land)*2 + o_bonus
    dict_steel[country] = len(land) + s_bonus
    dict_action[country] = len(land)
    if dict_people[country] >= 10:
        dict_people[country] += 1
    else:
        dict_people[country] = int((10-dict_people[country])*0.5+2)

print(f'第{t}回合开始时各国土地概况：',dict_land)
print(f'第{t}回合开始时各国剩余黄金概况：',dict_gold)
print(f'第{t}回合开始时各国剩余石油概况：',dict_oil)
print(f'第{t}回合开始时各国剩余钢铁概况：',dict_steel)
print(f'第{t}回合开始时各国战力概况：',dict_people)
print(f'第{t}回合开始时各国剩余行动点数概况：',dict_action)

### 战力部署阶段

In [ ]:
country_deploy = {}
land_deploy = {}
for country in clist:
    current_res = deploy(dict_land[country],country,dict_people[country])
    country_deploy[country] = current_res
    land_deploy.update(current_res)
    print(f"{country} 战力部署成功")

print('\n' + '-'*30)
print(f'第{t}回合各国土地部署情况:',country_deploy)
print(f'第{t}回合全球战力分布情况:',land_deploy)

### 行动阶段

In [ ]:
# 抽签决定行动顺序
order = input('输入抽签的行动顺序：(用英文逗号隔开)',)
order = [x.strip().upper() for x in order.split(',') if x.strip()]

In [ ]:
# 执行行动function
military(
    t=t, 
    order=order, 
    clist=clist, 
    dict_land=dict_land, 
    dict_action=dict_action, 
    dict_oil=dict_oil, 
    dict_steel=dict_steel, 
    country_deploy=country_deploy, 
    land_deploy=land_deploy, 
    dict_ceasefire=dict_ceasefire
)

### 谈判阶段

In [ ]:
#执行谈判function
diplomacy(
    t=t, 
    dict_gold=dict_gold, 
    dict_oil=dict_oil,
    dict_steel=dict_steel,
    dict_ceasefire=dict_ceasefire)

### 剩余行动点数兑换阶段

In [ ]:
# 执行行动点数兑换function
exchange_action(
    t=t,
    clist=clist,
    dict_action=dict_action,
    dict_gold=dict_gold,
    dict_oil=dict_oil,
    dict_steel=dict_steel)

### 回合结束结算阶段

In [ ]:
# 换算成黄金总量
for country in clist:
    dict_total[country] = dict_gold[country] + dict_oil[country]*10/8 + dict_steel[country]*10/5 + dict_action[country]*10 + len(dict_land[country])*20

# 回合结束后的资源概况
print("\n" + "="*30)
print(f'第{t}回合结束时各国土地概况：',dict_land)
print(f'第{t}回合结束时各国剩余黄金概况：',dict_gold)
print(f'第{t}回合结束时各国剩余石油概况：',dict_oil)
print(f'第{t}回合结束时各国剩余钢铁概况：',dict_steel)
print(f'第{t}回合结束时各国战力概况：',dict_people)
print(f'第{t}回合结束时各国剩余行动点数概况：',dict_action)
print(f'第{t}回合结束时各国资源换算成黄金总量情况：',dict_total)

### 保存本回合所有资源情况

In [ ]:
save_gss(t)

## 第6回合

In [ ]:
load_gss()
t = 6

### 回合开始阶段

In [ ]:
for country in clist:
    land = dict_land[country]
    g_bonus,o_bonus,s_bonus = bonus(land)
    dict_gold[country] = dict_gold[country] + g_bonus + dict_people[country]
    dict_oil[country] = len(land)*2 + o_bonus
    dict_steel[country] = len(land) + s_bonus
    dict_action[country] = len(land)
    if dict_people[country] >= 10:
        dict_people[country] += 1
    else:
        dict_people[country] = int((10-dict_people[country])*0.5+2)

print(f'第{t}回合开始时各国土地概况：',dict_land)
print(f'第{t}回合开始时各国剩余黄金概况：',dict_gold)
print(f'第{t}回合开始时各国剩余石油概况：',dict_oil)
print(f'第{t}回合开始时各国剩余钢铁概况：',dict_steel)
print(f'第{t}回合开始时各国战力概况：',dict_people)
print(f'第{t}回合开始时各国剩余行动点数概况：',dict_action)

### 战力部署阶段

In [ ]:
country_deploy = {}
land_deploy = {}
for country in clist:
    current_res = deploy(dict_land[country],country,dict_people[country])
    country_deploy[country] = current_res
    land_deploy.update(current_res)
    print(f"{country} 战力部署成功")

print('\n' + '-'*30)
print(f'第{t}回合各国土地部署情况:',country_deploy)
print(f'第{t}回合全球战力分布情况:',land_deploy)

### 行动阶段

In [ ]:
# 抽签决定行动顺序
order = input('输入抽签的行动顺序：(用英文逗号隔开)',)
order = [x.strip().upper() for x in order.split(',') if x.strip()]

In [ ]:
# 执行行动function
military(
    t=t, 
    order=order, 
    clist=clist, 
    dict_land=dict_land, 
    dict_action=dict_action, 
    dict_oil=dict_oil, 
    dict_steel=dict_steel, 
    country_deploy=country_deploy, 
    land_deploy=land_deploy, 
    dict_ceasefire=dict_ceasefire
)

### 谈判阶段

In [ ]:
#执行谈判function
diplomacy(
    t=t, 
    dict_gold=dict_gold, 
    dict_oil=dict_oil,
    dict_steel=dict_steel,
    dict_ceasefire=dict_ceasefire)

### 剩余行动点数兑换阶段

In [ ]:
# 执行行动点数兑换function
exchange_action(
    t=t,
    clist=clist,
    dict_action=dict_action,
    dict_gold=dict_gold,
    dict_oil=dict_oil,
    dict_steel=dict_steel)

### 回合结束结算阶段

In [ ]:
# 换算成黄金总量
for country in clist:
    dict_total[country] = dict_gold[country] + dict_oil[country]*10/8 + dict_steel[country]*10/5 + dict_action[country]*10 + len(dict_land[country])*20

# 回合结束后的资源概况
print("\n" + "="*30)
print(f'第{t}回合结束时各国土地概况：',dict_land)
print(f'第{t}回合结束时各国剩余黄金概况：',dict_gold)
print(f'第{t}回合结束时各国剩余石油概况：',dict_oil)
print(f'第{t}回合结束时各国剩余钢铁概况：',dict_steel)
print(f'第{t}回合结束时各国战力概况：',dict_people)
print(f'第{t}回合结束时各国剩余行动点数概况：',dict_action)
print(f'第{t}回合结束时各国资源换算成黄金总量情况：',dict_total)

### 保存本回合所有资源情况

In [ ]:
save_gss(t)

## 第7回合

In [ ]:
load_gss()
t = 7

### 回合开始阶段

In [ ]:
for country in clist:
    land = dict_land[country]
    g_bonus,o_bonus,s_bonus = bonus(land)
    dict_gold[country] = dict_gold[country] + g_bonus + dict_people[country]
    dict_oil[country] = len(land)*2 + o_bonus
    dict_steel[country] = len(land) + s_bonus
    dict_action[country] = len(land)
    if dict_people[country] >= 10:
        dict_people[country] += 1
    else:
        dict_people[country] = int((10-dict_people[country])*0.5+2)

print(f'第{t}回合开始时各国土地概况：',dict_land)
print(f'第{t}回合开始时各国剩余黄金概况：',dict_gold)
print(f'第{t}回合开始时各国剩余石油概况：',dict_oil)
print(f'第{t}回合开始时各国剩余钢铁概况：',dict_steel)
print(f'第{t}回合开始时各国战力概况：',dict_people)
print(f'第{t}回合开始时各国剩余行动点数概况：',dict_action)

### 战力部署阶段

In [ ]:
country_deploy = {}
land_deploy = {}
for country in clist:
    current_res = deploy(dict_land[country],country,dict_people[country])
    country_deploy[country] = current_res
    land_deploy.update(current_res)
    print(f"{country} 战力部署成功")

print('\n' + '-'*30)
print(f'第{t}回合各国土地部署情况:',country_deploy)
print(f'第{t}回合全球战力分布情况:',land_deploy)

### 行动阶段

In [ ]:
# 抽签决定行动顺序
order = input('输入抽签的行动顺序：(用英文逗号隔开)',)
order = [x.strip().upper() for x in order.split(',') if x.strip()]

In [ ]:
# 执行行动function
military(
    t=t, 
    order=order, 
    clist=clist, 
    dict_land=dict_land, 
    dict_action=dict_action, 
    dict_oil=dict_oil, 
    dict_steel=dict_steel, 
    country_deploy=country_deploy, 
    land_deploy=land_deploy, 
    dict_ceasefire=dict_ceasefire
)

### 谈判阶段

In [ ]:
#执行谈判function
diplomacy(
    t=t, 
    dict_gold=dict_gold, 
    dict_oil=dict_oil,
    dict_steel=dict_steel,
    dict_ceasefire=dict_ceasefire)

### 剩余行动点数兑换阶段

In [ ]:
# 执行行动点数兑换function
exchange_action(
    t=t,
    clist=clist,
    dict_action=dict_action,
    dict_gold=dict_gold,
    dict_oil=dict_oil,
    dict_steel=dict_steel)

### 回合结束结算阶段

In [ ]:
# 换算成黄金总量
for country in clist:
    dict_total[country] = dict_gold[country] + dict_oil[country]*10/8 + dict_steel[country]*10/5 + dict_action[country]*10 + len(dict_land[country])*20

# 回合结束后的资源概况
print("\n" + "="*30)
print(f'第{t}回合结束时各国土地概况：',dict_land)
print(f'第{t}回合结束时各国剩余黄金概况：',dict_gold)
print(f'第{t}回合结束时各国剩余石油概况：',dict_oil)
print(f'第{t}回合结束时各国剩余钢铁概况：',dict_steel)
print(f'第{t}回合结束时各国战力概况：',dict_people)
print(f'第{t}回合结束时各国剩余行动点数概况：',dict_action)
print(f'第{t}回合结束时各国资源换算成黄金总量情况：',dict_total)

### 保存本回合所有资源情况

In [ ]:
save_gss(t)

## 第8回合

In [ ]:
load_gss()
t = 8

### 回合开始阶段

In [ ]:
for country in clist:
    land = dict_land[country]
    g_bonus,o_bonus,s_bonus = bonus(land)
    dict_gold[country] = dict_gold[country] + g_bonus + dict_people[country]
    dict_oil[country] = len(land)*2 + o_bonus
    dict_steel[country] = len(land) + s_bonus
    dict_action[country] = len(land)
    if dict_people[country] >= 10:
        dict_people[country] += 1
    else:
        dict_people[country] = int((10-dict_people[country])*0.5+2)

print(f'第{t}回合开始时各国土地概况：',dict_land)
print(f'第{t}回合开始时各国剩余黄金概况：',dict_gold)
print(f'第{t}回合开始时各国剩余石油概况：',dict_oil)
print(f'第{t}回合开始时各国剩余钢铁概况：',dict_steel)
print(f'第{t}回合开始时各国战力概况：',dict_people)
print(f'第{t}回合开始时各国剩余行动点数概况：',dict_action)

### 战力部署阶段

In [ ]:
country_deploy = {}
land_deploy = {}
for country in clist:
    current_res = deploy(dict_land[country],country,dict_people[country])
    country_deploy[country] = current_res
    land_deploy.update(current_res)
    print(f"{country} 战力部署成功")

print('\n' + '-'*30)
print(f'第{t}回合各国土地部署情况:',country_deploy)
print(f'第{t}回合全球战力分布情况:',land_deploy)

### 行动阶段

In [ ]:
# 抽签决定行动顺序
order = input('输入抽签的行动顺序：(用英文逗号隔开)',)
order = [x.strip().upper() for x in order.split(',') if x.strip()]

In [ ]:
# 执行行动function
military(
    t=t, 
    order=order, 
    clist=clist, 
    dict_land=dict_land, 
    dict_action=dict_action, 
    dict_oil=dict_oil, 
    dict_steel=dict_steel, 
    country_deploy=country_deploy, 
    land_deploy=land_deploy, 
    dict_ceasefire=dict_ceasefire
)

### 谈判阶段(最后一回合，没有该阶段)

### 剩余行动点数兑换阶段

In [ ]:
# 执行行动点数兑换function
exchange_action(
    t=t,
    clist=clist,
    dict_action=dict_action,
    dict_gold=dict_gold,
    dict_oil=dict_oil,
    dict_steel=dict_steel)

### 回合结束结算阶段

In [ ]:
# 换算成黄金总量
for country in clist:
    dict_total[country] = dict_gold[country] + dict_oil[country]*10/8 + dict_steel[country]*10/5 + dict_action[country]*10 + len(dict_land[country])*20

# 回合结束后的资源概况
print("\n" + "="*30)
print(f'第{t}回合结束时各国土地购买概况：',dict_land)
print(f'第{t}回合结束时各国剩余黄金概况：',dict_gold)
print(f'第{t}回合结束时各国剩余石油概况：',dict_oil)
print(f'第{t}回合结束时各国剩余钢铁概况：',dict_steel)
print(f'第{t}回合结束时各国战力概况：',dict_people)
print(f'第{t}回合结束时各国剩余行动点数概况：',dict_action)
print(f'第{t}回合结束时各国资源换算成黄金总量情况：',dict_total)